## LAB | Prompt Engineering Lab

You've been hired as an AI consultant by TechFlow Solutions, a company that's struggling with their customer service chatbot. The chatbot is producing inconsistent, unreliable responses that are frustrating customers and hurting the company's reputation. Your task is to diagnose the problems with their current prompts and systematically improve them using professional prompt engineering techniques.

### Part 1: Starting Simple - Create Initial Prompts

#### Step 1: Setup Your Environment

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from openai import OpenAI

# OpenAI client setup
client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")


def run_prompt(prompt, model=MODEL):
    """Send one prompt to the OpenAI Responses API and return plain text."""
    response = client.responses.create(
        model=model,
        input=prompt,
    )
    return response.output_text.strip()


def run_prompts_parallel(prompts, model=MODEL, max_workers=5):
    """Run multiple prompts concurrently and return results in the same order."""
    prompts = list(prompts)
    if not prompts:
        return []

    workers = min(max_workers, len(prompts))
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(run_prompt, prompt, model) for prompt in prompts]
        return [future.result() for future in futures]


# Smoke test for the checkpoint
_ = run_prompt("Reply with exactly: Setup complete")
print("Setup complete")


Setup complete


#### Step 2: Create Initial Prompts

In [2]:
# Task 1: Sentiment Analysis

# Initial simple prompt for sentiment analysis
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""
 
# Test it once
result = run_prompt(sentiment_prompt_v1)
print("Sentiment Analysis Result:")
print(result)

Sentiment Analysis Result:
Positive


In [3]:
# Task 2: Product Description Generation

# Initial simple prompt for product description
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""
 
# Test it once
result = run_prompt(product_prompt_v1)
print("Product Description Result:")
print(result)

Product Description Result:
Meet the Wireless Mouse, designed for smooth, reliable performance at an affordable price of just **$29.99**. With a comfortable ergonomic shape and responsive tracking, it’s ideal for work, school, or everyday use.

**Features:**
- **Wireless convenience** for a clutter-free setup
- **Smooth, accurate tracking** for easy navigation
- **Comfortable design** for all-day use
- **Simple plug-and-play setup**
- **Great value** at only **$29.99**

Whether you're upgrading your desk or need a dependable mouse on the go, this wireless mouse delivers everyday performance without the hassle of cords.


In [4]:
# Task 3: Data Extraction

# Initial simple prompt for data extraction
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""
 
# Test it once
result = run_prompt(extraction_prompt_v1)
print("Data Extraction Result:")
print(result)

Data Extraction Result:
Here’s the extracted information:

- **Order item:** #12345  
- **Order date:** March 15th  
- **Delivery feedback:** Fast  
- **Packaging feedback:** Damaged


### Part 2: Diagnosing Failures - Systematic Testing

#### Step 3: Run Prompts 5 Times

In [5]:
from collections import defaultdict
import pandas as pd

def run_prompt_n_times(prompt, n=5, model=MODEL):
    """Run the same prompt multiple times and return all outputs."""
    return [run_prompt(prompt, model=model) for _ in range(n)]

def test_prompts_5_times(prompt_dict, n=5, model=MODEL):
    """
    prompt_dict format:
    {
        "sentiment": "prompt text",
        "classification": "prompt text",
        "summary": "prompt text"
    }
    """
    results = defaultdict(list)

    for name, prompt in prompt_dict.items():
        results[name] = run_prompt_n_times(prompt, n=n, model=model)

    return results

# Example prompts from your lab
prompt_dict = {
    "sentiment": sentiment_prompt_v1,
    "product": product_prompt_v1,
    "extraction": extraction_prompt_v1
}

results = test_prompts_5_times(prompt_dict, n=5)

for task, outputs in results.items():
    print(f"\n=== {task.upper()} ===")
    for i, output in enumerate(outputs, start=1):
        print(f"Run {i}: {output}")



=== SENTIMENT ===
Run 1: Positive
Run 2: Positive
Run 3: **Classification:** Positive / Praise

The customer is expressing satisfaction and appreciation for the product.
Run 4: Sentiment: **Positive**
Run 5: positive

=== PRODUCT ===
Run 1: Upgrade your everyday computing with this sleek wireless mouse, designed for comfort, precision, and convenience. Its responsive tracking and reliable wireless connection help you work and browse with ease, while the ergonomic shape fits naturally in your hand for all-day use.

At just **$29.99**, it’s an affordable way to add smooth performance and clutter-free control to your desk. Compact, dependable, and easy to set up, this wireless mouse is a smart choice for home, office, or on-the-go use.
Run 2: Meet the sleek Wireless Mouse, designed for smooth everyday performance at just **$29.99**. With reliable wireless connectivity, comfortable ergonomic shaping, and precise cursor control, it’s a great choice for work, study, or casual browsing.

**H

In [6]:
for task, outputs in results.items():
    unique_outputs = len(set(outputs))
    print(f"{task}: {unique_outputs}/5 unique responses")

sentiment: 4/5 unique responses
product: 5/5 unique responses
extraction: 5/5 unique responses


**Observations:** We can see that the vast majority of outputs are unique, 13/15 or 86.66%. We can also assess that repetitions only take place in the sentiment outputs, and that sentiment analysis and data extraction do not vary greatly content-wise, the changes are merely simple format differences, as expected. For product description generation instead we do see that content varies and the tool gets creative. Some decide to show the features and have more sober catching phrases without mentionning features like Run 3, or more intense and cumbersome as Run 4, which could be classified as different marketing perspectives. 

#### Step 4: Run Prompts 10 Times

In [7]:
import pandas as pd

prompts = {
    "sentiment": sentiment_prompt_v1,
    "product_description": product_prompt_v1,
    "data_extraction": extraction_prompt_v1,
}

def run_prompt_n_times(prompt, n=10, model=MODEL):
    return [run_prompt(prompt, model=model) for _ in range(n)]

results_5 = {name: run_prompt_n_times(prompt, n=5) for name, prompt in prompts.items()}
results_10 = {name: run_prompt_n_times(prompt, n=10) for name, prompt in prompts.items()}

rows = []

for name in prompts:
    for i, output in enumerate(results_10[name], start=1):
        rows.append({
            "prompt": name,
            "test_size": 10,
            "run": i,
            "response": output
        })

df = pd.DataFrame(rows)

# Display the full table for better readability
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

df


,prompt,test_size,run,response
0,sentiment,10,1,Positive
1,sentiment,10,2,positive
2,sentiment,10,3,**Sentiment:** Positive
3,sentiment,10,4,**Positive**
4,sentiment,10,5,Positive
5,sentiment,10,6,Positive
6,sentiment,10,7,Positive
7,sentiment,10,8,**Sentiment:** Positive \n**Category:** Praise / Satisfaction
8,sentiment,10,9,Positive
9,sentiment,10,10,Positive


#### Step 5: Run Prompts 15 Times and Create Failure Analysis

In [8]:
import pandas as pd
from collections import Counter

# Step 5: Run Prompts 15 Times and Create Failure Analysis

prompts = {
    "sentiment": sentiment_prompt_v1,
    "product_description": product_prompt_v1,
    "data_extraction": extraction_prompt_v1,
}

def run_prompt_n_times(prompt, n=15, model=MODEL):
    return [run_prompt(prompt, model=model) for _ in range(n)]

results_15 = {
    name: run_prompt_n_times(prompt, n=15)
    for name, prompt in prompts.items()
}

rows = []

for name, outputs in results_15.items():
    counts = Counter(outputs)
    total_runs = len(outputs)
    unique_responses = len(counts)
    most_common_response, most_common_count = counts.most_common(1)[0]
    consistency_pct = round((most_common_count / total_runs) * 100, 1)

    rows.append({
        "prompt": name,
        "total_runs": total_runs,
        "unique_responses": unique_responses,
        "consistency_pct": consistency_pct,
        "most_common_response": most_common_response,
    })

failure_analysis_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

failure_analysis_df


,prompt,total_runs,unique_responses,consistency_pct,most_common_response
0,sentiment,15,9,33.3,Positive
1,product_description,15,15,6.7,"Meet the **Wireless Mouse**, designed for smooth everyday performance and effortless comfort. With a responsive connection, precise tracking, and a sleek ergonomic shape, it’s perfect for work, school, or casual browsing. Compact and easy to carry, this mouse delivers reliable wireless convenience without the clutter of cords.\n\n**Price:** **$29.99**"
2,data_extraction,15,10,33.3,Here’s the extracted information:\n\n- **Order item:** #12345 \n- **Order date:** March 15th \n- **Delivery feedback:** Fast \n- **Packaging feedback:** Damaged


In [9]:
# Printing the outputs to eyeball patterns
for prompt_name, outputs in results_15.items():
    print(f"\n{'=' * 20} {prompt_name.upper()} {'=' * 20}")
    for i, output in enumerate(outputs, start=1):
        print(f"\nRun {i}:\n{output}\n")


==================== SENTIMENT ====================

Run 1:
Positive


Run 2:
**Category:** Positive feedback / praise


Run 3:
The message is **positive sentiment**.


Run 4:
**Sentiment:** Positive


Run 5:
**Category:** Positive feedback / Praise


Run 6:
Positive sentiment


Run 7:
Positive sentiment


Run 8:
Positive


Run 9:
**Classification:** Positive feedback / praise


Run 10:
**Sentiment:** Positive  
**Category:** Praise / Satisfaction


Run 11:
Positive


Run 12:
Positive


Run 13:
**Classification:** Positive sentiment / satisfied customer

The customer is expressing strong satisfaction and approval of the product.


Run 14:
**Sentiment:** Positive


Run 15:
Positive


==================== PRODUCT_DESCRIPTION ====================

Run 1:
Meet the **Wireless Mouse**, designed for smooth everyday performance and effortless comfort. With a responsive connection, precise tracking, and a sleek ergonomic shape, it’s perfect for work, school, or casual browsing. Compact and eas

##### Failure patterns

**Sentiment:** Inconsistent formatting. Unnecessary categorization or overexplanation of the sentiment a consumer would not need. No hallucination or content variation.
#
**Product description:** Inconsistent formatting. Confusing price as features. Repetition of words in content. Length inconsistency. Inconsistent style
#
**Data extraction:** Inconsistent formatting. Adding commentary on what to do with the data. However pretty consistent fields name.

### Part 3: Iteration 1 - Rewriting Simple Prompts

#### Step 6: Improve Sentiment Analysis Prompt

In [10]:
# Task 1: Sentiment Analysis

# Initial simple prompt for sentiment analysis
sentiment_prompt_v2 = """
Classify this customer message: "I love this product! It's exactly what I needed."
The format should be: Sentiment: [Positive/Negative/Neutral]
2 words maximum for the output.
"""
 
# Test it once
result = run_prompt(sentiment_prompt_v2)
print("Sentiment Analysis Result:")
print(result)

Sentiment Analysis Result:
Sentiment: Positive


#### Step 7: Improve Product Description Prompt

In [11]:
# Task 2: Product Description Generation

# Initial simple prompt for product description
product_prompt_v2 = """
Create a product description for a wireless mouse that costs $29.99.
The format should be: 
[Product Description in sentences]
[3 Key Features in bullet points]
[2 Benefits in slogan style sentences]
These categories should not be labeled in the output, just formatted as described.
The description should be concise and highlight most important aspects. It should be no more than 2 sentences.
The style should be sober and persuasive, targeting tech-savvy consumers looking for affordable accessories.
The output should be no more than 30 words.
"""
 
# Test it once
result = run_prompt(product_prompt_v2)
print("Product Description Result:")
print(result)

Product Description Result:
Reliable wireless mouse for precise, clutter-free control at an affordable $29.99.  
- 2.4GHz wireless connection  
- Compact, ergonomic design  
- Long battery life  
Smooth tracking. Portable convenience.


#### Step 8: Improve Data Extraction Prompt:

In [12]:
# Task 3: Data Extraction

# Initial simple prompt for data extraction
extraction_prompt_v2 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
The format should be:
- Order Number: [Extracted order number]
- Order Date: [Extracted order date]
- Delivery Feedback: [Extracted delivery feedback]
- Packaging Feedback: [Extracted packaging feedback]
All responses should start with a capital letter and be 2 words maximum.
There should be no additional text or labels, just the extracted information formatted as described.
"""
 
# Test it once
result = run_prompt(extraction_prompt_v2)
print("Data Extraction Result:")
print(result)

Data Extraction Result:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


### Part 4: Iteration 2 - Adding Structure & Constraints

#### Step 9: Add Few-Shot Examples to Sentiment Analysis

In [13]:
import pandas as pd
from collections import Counter

# Run Prompt with Few-shot examples 15 Times and track consistency 

sentiment_prompt_v2 = """
Classify this customer message: "I love this product! It's exactly what I needed."
The format should be: Sentiment: [Positive/Negative/Neutral]
2 words maximum for the output.

Follow the examples below for formatting and style:

Example 1:
Sentiment: Positive

Example 2:
Sentiment: Negative

Example 3:
Sentiment: Neutral

Now classify the sentiment of the input provided. If none is provided, use the default example above.
"""

prompts = {
    "sentiment": sentiment_prompt_v2,
}

def run_prompt_n_times(prompt, n=15, model=MODEL):
    return [run_prompt(prompt, model=model) for _ in range(n)]

results_15 = {
    name: run_prompt_n_times(prompt, n=15)
    for name, prompt in prompts.items()
}

rows = []

for name, outputs in results_15.items():
    counts = Counter(outputs)
    total_runs = len(outputs)
    unique_responses = len(counts)
    most_common_response, most_common_count = counts.most_common(1)[0]
    consistency_pct = round((most_common_count / total_runs) * 100, 1)

    rows.append({
        "prompt": name,
        "total_runs": total_runs,
        "unique_responses": unique_responses,
        "consistency_pct": consistency_pct,
        "most_common_response": most_common_response,
    })

failure_analysis_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

failure_analysis_df

,prompt,total_runs,unique_responses,consistency_pct,most_common_response
0,sentiment,15,1,100.0,Sentiment: Positive


#### Step 10: Add Chain-of-Thought to Data Extraction

In [14]:

import pandas as pd
from collections import Counter

# Run Prompt with Few-shot examples 15 Times and track consistency 

extraction_prompt_v2 = """
Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged
You are a careful data extraction assistant.

Analyze the text step by step to identify all relevant facts, but only return the final extracted data in the exact format below.

- Order Number: [Extracted order number]
- Order Date: [Extracted order date]
- Delivery Feedback: [Extracted delivery feedback]
- Packaging Feedback: [Extracted packaging feedback]

All responses should start with a capital letter and be 2 words maximum.
There should be no additional text or labels, just the extracted information formatted as described.

Now extract information from the input provided.
"""

prompts = {
    "extraction": extraction_prompt_v2,
}

def run_prompt_n_times(prompt, n=15, model=MODEL):
    return [run_prompt(prompt, model=model) for _ in range(n)]

results_15 = {
    name: run_prompt_n_times(prompt, n=15)
    for name, prompt in prompts.items()
}

rows = []

for name, outputs in results_15.items():
    counts = Counter(outputs)
    total_runs = len(outputs)
    unique_responses = len(counts)
    most_common_response, most_common_count = counts.most_common(1)[0]
    consistency_pct = round((most_common_count / total_runs) * 100, 1)

    rows.append({
        "prompt": name,
        "total_runs": total_runs,
        "unique_responses": unique_responses,
        "consistency_pct": consistency_pct,
        "most_common_response": most_common_response,
    })

for i, output in enumerate(results_15["extraction"], start=1):
    print(f"\nRun {i}:\n{output}\n")

failure_analysis_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

failure_analysis_df


Run 1:
- Order Number: #12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


Run 2:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


Run 3:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


Run 4:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


Run 5:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


Run 6:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


Run 7:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


Run 8:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: Damaged


Run 9:
- Order Number: 12345
- Order Date: March 15th
- Delivery Feedback: Fast
- Packaging Feedback: 

,prompt,total_runs,unique_responses,consistency_pct,most_common_response
0,extraction,15,2,93.3,- Order Number: 12345\n- Order Date: March 15th\n- Delivery Feedback: Fast\n- Packaging Feedback: Damaged


#### Step 11: Add Few-Shot Examples and Structure to Product Description


In [15]:
# Add Few-Shot Examples and Structure to Product Description

product_prompt_v2 = """
You are a professional copywriter.

Create a product description for a wireless mouse that costs $29.99.
The format should be: 
[Product Description in sentences]
[3 Key Features in bullet points]
[2 Benefits in slogan style sentences]
These categories should not be labeled in the output, just formatted as described.
The description should be concise and highlight most important aspects. It should be no more than 2 sentences.
The style should be sober and persuasive, targeting tech-savvy consumers looking for affordable accessories.
The output should be no more than 40 words.

Follow the style of these examples:

Example 1:
Reliable wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and budget-friendly.

- Wireless freedom
- Accurate tracking
- Affordable value

Work smarter, spend less.  
Smooth performance, effortless setup.

Example 2:
Wireless mouse for precise everyday control, priced at $29.99. Compact, smart, and built for efficient productivity.  

- Wireless connectivity  
- Ergonomic design  
- Long battery life  

Performance without clutter. Value without compromise.

Now write a product description for the input provided. If none is provided, use the default example above.
"""

# Test version 2 fifteen times
product_v2_results_15 = run_prompt_n_times(product_prompt_v2, n=15)

for i, output in enumerate(product_v2_results_15, start=1):
    print(f"\nRun {i}:\n{output}\n")

prompts = {
    "product": product_prompt_v2,
}

def run_prompt_n_times(prompt, n=15, model=MODEL):
    return [run_prompt(prompt, model=model) for _ in range(n)]

results_15 = {
    name: run_prompt_n_times(prompt, n=15)
    for name, prompt in prompts.items()
}

rows = []

for name, outputs in results_15.items():
    counts = Counter(outputs)
    total_runs = len(outputs)
    unique_responses = len(counts)
    most_common_response, most_common_count = counts.most_common(1)[0]
    consistency_pct = round((most_common_count / total_runs) * 100, 1)

    rows.append({
        "prompt": name,
        "total_runs": total_runs,
        "unique_responses": unique_responses,
        "consistency_pct": consistency_pct,
        "most_common_response": most_common_response,
    })

failure_analysis_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

failure_analysis_df



Run 1:
Wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and built for efficient productivity.

- Wireless connectivity
- Accurate tracking
- Affordable value

Performance without clutter.  
Smooth control, smart savings.


Run 2:
Wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and built for efficient productivity.

- Wireless connectivity
- Accurate tracking
- Affordable value

Performance without clutter. Value without compromise.


Run 3:
Wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and built for efficient productivity.

- Wireless connectivity
- Accurate tracking
- Affordable value

Performance without clutter.  
Value without compromise.


Run 4:
Reliable wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and budget-friendly.

- Wireless connectivity
- Precise tracking
- Affordable value

Work smarter, spend less.  
Smooth control, clean se

,prompt,total_runs,unique_responses,consistency_pct,most_common_response
0,product,15,11,26.7,"Wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and built for efficient productivity.\n\n- Wireless connectivity\n- Accurate tracking\n- Affordable value\n\nPerformance without clutter. Value without compromise."


**Comparison of v3 with v1 results:**
#
v1 overall consistency: 18.8%
#
v3 overall consistency: 71,1%

In version 3 **sentiment and data extraction** resolved all failure patterns by reaching 1 unique response per prompt and 100% consistency
**product description** improved considerably, more than doubled consistency but we can still see the same failure patterns except for style inconsistency.

### Part 5: Tuning for Different Tasks

#### Step 12: Create Task Variations

In [16]:
# Create Task Variations

task_variations = {
    "sentiment": [
        "I love this product! It's exactly what I needed.",
        "This is okay, but I expected more.",
        "I'm really disappointed and want a refund.",
    ],
    "product_description": [
        "Create a product description for a wireless mouse that costs $29.99.",
        "Create a product description for a gaming mouse that costs $59.99.",
        "Create a product description for an ergonomic wireless mouse for office use that costs $39.99.",
    ],
    "data_extraction": [
        "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.",
        "I bought item #67890 on April 2nd. Shipping was delayed and the box was torn.",
        "I placed order #24680 on May 1st. Delivery was on time and packaging was intact.",
    ],
}

best_prompts = {
    "sentiment": """Classify this customer message: {input text}
The format should be: Sentiment: [Positive/Negative/Neutral]
2 words maximum for the output.

Follow the examples below for formatting and style:

Example 1:
Sentiment: Positive

Example 2:
Sentiment: Negative

Example 3:
Sentiment: Neutral

Now classify the sentiment of the input provided. If none is provided, use the default example above.""",
    "product_description": """
You are a professional copywriter.

The format should be: 
[Product Description in sentences]
[3 Key Features in bullet points]
[2 Benefits in slogan style sentences]
These categories should not be labeled in the output, just formatted as described.
The description should be concise and highlight most important aspects. It should be no more than 2 sentences.
The style should be sober and persuasive, targeting tech-savvy consumers looking for affordable accessories.
The output should be no more than 40 words.

Follow the style of these examples:

Example 1:
Reliable wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and budget-friendly.

- Wireless freedom
- Accurate tracking
- Affordable value

Work smarter, spend less.  
Smooth performance, effortless setup.

Example 2:
Wireless mouse for precise everyday control, priced at $29.99. Compact, smart, and built for efficient productivity.  

- Wireless connectivity  
- Ergonomic design  
- Long battery life  

Performance without clutter. Value without compromise.

Now write a product description for the input provided. If none is provided, use the default example above.
""",
    "data_extraction": """
You are a careful data extraction assistant.

Analyze the input text step by step to identify all relevant facts, but only return the final extracted data in the exact format below.

- Order Number: [Extracted order number]
- Order Date: [Extracted order date]
- Delivery Feedback: [Extracted delivery feedback]
- Packaging Feedback: [Extracted packaging feedback]

All responses should start with a capital letter and be 2 words maximum.
There should be no additional text or labels, just the extracted information formatted as described.

Now extract information from the input text provided.
""",
}

variation_results = {}

for task_name, prompt_template in best_prompts.items():
    variation_results[task_name] = []
    for variation in task_variations[task_name]:
        output = run_prompt(prompt_template.replace("{input text}", variation))
        variation_results[task_name].append({
            "input": variation,
            "output": output
        })

for task_name, results in variation_results.items():
    print(f"\n=== {task_name.upper()} ===")
    for item in results:
        print(f"\nInput: {item['input']}\nOutput: {item['output']}\n")


=== SENTIMENT ===

Input: I love this product! It's exactly what I needed.
Output: Sentiment: Positive


Input: This is okay, but I expected more.
Output: Sentiment: Negative


Input: I'm really disappointed and want a refund.
Output: Sentiment: Negative


=== PRODUCT_DESCRIPTION ===

Input: Create a product description for a wireless mouse that costs $29.99.
Output: Reliable wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and budget-friendly.

- Wireless freedom
- Accurate tracking
- Affordable value

Work smarter, spend less.  
Smooth performance, effortless setup.


Input: Create a product description for a gaming mouse that costs $59.99.
Output: Reliable wireless mouse for precise everyday control, priced at $29.99. Compact, responsive, and budget-friendly.

- Wireless freedom
- Accurate tracking
- Affordable value

Work smarter, spend less.  
Smooth performance, effortless setup.


Input: Create a product description for an ergonomic wireless m

**Sentiment** works through the examples I set initially because they're varied.
#
however **Product description** gets anchored on the examples I gave for the 29.99 mouse from before.
#
and **Data extraction** does not even recognise the new input, even after deleting the examples.

#### Step 13: Final Evaluation and Comparison

In [18]:
# Defining final v3 prompts

sentiment_prompt_v3 = """
You are a customer support analyst.

Classify the sentiment of the customer message below.

Return only one label:
Positive, Neutral, or Negative.

The format should be: Sentiment: [Positive/Negative/Neutral]
2 words maximum for the output.

Follow the examples below for formatting and style:

Example 1:
Sentiment: Positive

Example 2:
Sentiment: Negative

Example 3:
Sentiment: Neutral

Customer message:
{input_text}
"""

product_prompt_v3 = """
You are a professional copywriter.

Write a product description in this exact format:

Title: ...
Description: ...
Features (only 3, in bullet points):
- ...
- ...
- ...
Price: ...

The style should be sober and persuasive, targeting tech-savvy consumers looking for affordable accessories.
The output should be no more than 40 words.

Product request:
{input_text}
"""

extraction_prompt_v3 = """
You are a precise information extraction assistant.

Extract the requested fields from the text below.

Analyze the input text step by step to identify all relevant facts, but only return the final extracted data in the exact format below.

- Order Number: [Extracted order number]
- Order Date: [Extracted order date]
- Delivery Feedback: [Extracted delivery feedback]
- Packaging Feedback: [Extracted packaging feedback]

All responses should start with a capital letter and be 2 words maximum.
There should be no additional text or labels, just the extracted information formatted as described.

Text:
{input_text}
"""

evaluation_inputs = {
    "sentiment": [
        "I love this product! It's exactly what I needed.",
        "This is fine, but not amazing.",
        "I'm really disappointed and want a refund.",
    ],
    "product_description": [
        "Create a product description for a wireless mouse that costs $29.99.",
        "Create a product description for a gaming mouse that costs $59.99.",
        "Create a product description for an ergonomic wireless mouse for office use that costs $39.99.",
    ],
    "data_extraction": [
        "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.",
        "I bought item #67890 on April 2nd. Shipping was delayed and the box was torn.",
        "I placed order #24680 on May 1st. Delivery was on time and packaging was intact.",
    ],
}

from collections import Counter
import pandas as pd

v3_prompts = {
    "sentiment": sentiment_prompt_v3,
    "product_description": product_prompt_v3,
    "data_extraction": extraction_prompt_v3,
}

def run_template_n_times(prompt_template, input_text, n=15, model=MODEL):
    prompt = prompt_template.format(input_text=input_text)
    return [run_prompt(prompt, model=model) for _ in range(n)]

def summarize_outputs(outputs):
    counts = Counter(outputs)
    most_common_response, most_common_count = counts.most_common(1)[0]
    return {
        "total_runs": len(outputs),
        "unique_responses": len(counts),
        "consistency_pct": round((most_common_count / len(outputs)) * 100, 1),
        "most_common_response": most_common_response,
    }

report_rows = []

for task_name, prompt_template in v3_prompts.items():
    for input_text in evaluation_inputs[task_name]:
        outputs = run_template_n_times(prompt_template, input_text, n=15)
        summary = summarize_outputs(outputs)

        report_rows.append({
            "task": task_name,
            "input": input_text,
            **summary,
        })

# Comparison of outputs across tasks and inputs

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

def summarize_outputs(outputs):
    counts = Counter(outputs)
    most_common_response, most_common_count = counts.most_common(1)[0]
    return {
        "total_runs": len(outputs),
        "unique_responses": len(counts),
        "consistency_pct": round((most_common_count / len(outputs)) * 100, 1),
        "most_common_count": most_common_count,
        "most_common_response": most_common_response,
    }

# Per-input report
report_rows = []
task_all_outputs = {}

for task_name, prompt_template in v3_prompts.items():
    task_all_outputs[task_name] = []

    for input_text in evaluation_inputs[task_name]:
        outputs = run_template_n_times(prompt_template, input_text, n=15)
        task_all_outputs[task_name].extend(outputs)

        summary = summarize_outputs(outputs)
        report_rows.append({
            "task": task_name,
            "input_text": input_text,
            **summary,
        })

final_report_df = pd.DataFrame(report_rows)

# Task-level summary across all inputs
task_summary_rows = []
for task_name, outputs in task_all_outputs.items():
    counts = Counter(outputs)
    most_common_response, most_common_count = counts.most_common(1)[0]

    task_summary_rows.append({
        "task": task_name,
        "inputs_tested": len(evaluation_inputs[task_name]),
        "total_runs": len(outputs),
        "unique_responses_across_all_inputs": len(counts),
        "most_common_count": most_common_count,
        "overall_consistency_pct": round((most_common_count / len(outputs)) * 100, 1),
        "most_common_response_across_all_inputs": most_common_response,
    })

task_summary_df = pd.DataFrame(task_summary_rows)

print("Per-input report:")
display(final_report_df)

print("\nTask-level summary:")
display(task_summary_df)


Per-input report:


,task,input_text,total_runs,unique_responses,consistency_pct,most_common_count,most_common_response
0,sentiment,I love this product! It's exactly what I needed.,15,1,100.0,15,Sentiment: Positive
1,sentiment,"This is fine, but not amazing.",15,1,100.0,15,Sentiment: Neutral
2,sentiment,I'm really disappointed and want a refund.,15,1,100.0,15,Sentiment: Negative
3,product_description,Create a product description for a wireless mouse that costs $29.99.,15,15,6.7,1,"Title: Wireless Mouse \nDescription: Reliable, comfortable, and built for daily productivity, this wireless mouse delivers smooth control without clutter. \nFeatures (only 3, in bullet points):\n- 2.4GHz wireless connection\n- Ergonomic compact design\n- Long battery life\nPrice: $29.99"
4,product_description,Create a product description for a gaming mouse that costs $59.99.,15,15,6.7,1,"Title: Precision Gaming Mouse \nDescription: Responsive, reliable, and built for competitive play, this gaming mouse delivers smooth control and comfort at an accessible price. \nFeatures (only 3, in bullet points):\n- Adjustable DPI for accurate tracking\n- Ergonomic design for long sessions\n- Durable switches for lasting performance \nPrice: $59.99"
5,product_description,Create a product description for an ergonomic wireless mouse for office use that costs $39.99.,15,15,6.7,1,"Title: Ergonomic Wireless Office Mouse \nDescription: Smooth, comfortable, and built for all-day productivity, this wireless mouse delivers precise control with a compact, modern design. \nFeatures (only 3, in bullet points):\n- Ergonomic shape for reduced strain\n- Reliable wireless connectivity\n- Precise tracking for office tasks\nPrice: $39.99"
6,data_extraction,I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.,15,1,100.0,15,- Order Number: 12345\n- Order Date: March 15th\n- Delivery Feedback: Fast\n- Packaging Feedback: Damaged
7,data_extraction,I bought item #67890 on April 2nd. Shipping was delayed and the box was torn.,15,2,73.3,11,- Order Number: 67890\n- Order Date: April 2nd\n- Delivery Feedback: Shipping delayed\n- Packaging Feedback: Box torn
8,data_extraction,I placed order #24680 on May 1st. Delivery was on time and packaging was intact.,15,1,100.0,15,- Order Number: 24680\n- Order Date: May 1st\n- Delivery Feedback: On time\n- Packaging Feedback: Intact



Task-level summary:


,task,inputs_tested,total_runs,unique_responses_across_all_inputs,most_common_count,overall_consistency_pct,most_common_response_across_all_inputs
0,sentiment,3,45,3,15,33.3,Sentiment: Positive
1,product_description,3,45,45,1,2.2,"Title: Wireless Mouse \nDescription: Reliable, comfortable, and built for daily productivity, this wireless mouse delivers smooth control without clutter. \nFeatures (only 3, in bullet points):\n- 2.4GHz wireless connection\n- Ergonomic compact design\n- Long battery life\nPrice: $29.99"
2,data_extraction,3,45,4,15,33.3,- Order Number: 12345\n- Order Date: March 15th\n- Delivery Feedback: Fast\n- Packaging Feedback: Damaged


In [19]:
comparison_notes = [
    {
        "task": "sentiment",
        "improvement": "Perfect consistency score, cleaner label-only outputs.",
        "remaining_issue": "None detected."
    },
    {
        "task": "product_description",
        "improvement": "More stable structure and style across input variations.",
        "remaining_issue": "May need tighter wording for audience-specific tone. Also format and length stricter constraints may allow more consistency in output structure and content."
    },
    {
        "task": "data_extraction",
        "improvement": "Better field consistency and cleaner extraction.",
        "remaining_issue": "May need stricter formatting for edge cases (2 unique responses in one task)."
    },
]

pd.DataFrame(comparison_notes)


,task,improvement,remaining_issue
0,sentiment,"Perfect consistency score, cleaner label-only outputs.",None detected.
1,product_description,More stable structure and style across input variations.,May need tighter wording for audience-specific tone. Also format and length stricter constraints may allow more consistency in output structure and content.
2,data_extraction,Better field consistency and cleaner extraction.,May need stricter formatting for edge cases (2 unique responses in one task).
